#**1_data_ingestion**

##**Install Dependencies**

In [1]:
!pip uninstall -y datasets
!pip install datasets==2.14.7
!pip install pyspark
!pip install pyarrow

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.4/520.4 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 18.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
  Attempting uninstall: mu

##**Spark Session Configuration**

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviewsBigData") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

spark

##**Load Dataset (Streaming)**

In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Electronics",
    split="full",
    streaming=True
)

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


##**Sample ~1.5 Million Records**

In [4]:
import itertools
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

batch = list(itertools.islice(dataset, 200000))  # take 200k rows first
pdf = pd.DataFrame(batch)
spark_df = spark.createDataFrame(pdf)

spark_df.show(5)

+------+--------------------+--------------------+--------------------+----------+-----------+--------------------+-------------+------------+-----------------+
|rating|               title|                text|              images|      asin|parent_asin|             user_id|    timestamp|helpful_vote|verified_purchase|
+------+--------------------+--------------------+--------------------+----------+-----------+--------------------+-------------+------------+-----------------+
|   3.0|Smells like gasol...|First & most offe...|[{large_image_url...|B083NRGZMM| B083NRGZMM|AFKZENTNBQ7A7V7UX...|1658185117948|           0|             true|
|   1.0|Didn’t work at al...|These didn’t work...|                  []|B07N69T6TM| B07N69T6TM|AFKZENTNBQ7A7V7UX...|1592678549731|           0|             true|
|   5.0|          Excellent!|I love these. The...|                  []|B01G8JO5F2| B01G8JO5F2|AFKZENTNBQ7A7V7UX...|1523093017534|           0|             true|
|   5.0|Great laptop back...|I was

##**Validate Big Data Size**

In [5]:
print("Total Rows:", spark_df.count())
spark_df.printSchema()

Total Rows: 200000
root
 |-- rating: double (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: string
 |    |    |-- value: string (valueContainsNull = true)
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



##**Repartition for Parallelism**

In [6]:
spark_df = spark_df.repartition(200)
print("Partitions:", spark_df.rdd.getNumPartitions())

Partitions: 200


In [7]:
spark_df.write.mode("overwrite").parquet("data/amazon_reviews_parquet")

#**2_feature_engineering**

##**Load Data**

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()

df = spark.read.parquet("data/amazon_reviews_parquet")
df.show(5)

+------+--------------------+--------------------+------+----------+-----------+--------------------+-------------+------------+-----------------+
|rating|               title|                text|images|      asin|parent_asin|             user_id|    timestamp|helpful_vote|verified_purchase|
+------+--------------------+--------------------+------+----------+-----------+--------------------+-------------+------------+-----------------+
|   5.0|A lot of safety i...|Better to be safe...|    []|B075XGZ6GL| B07XPQF1FJ|AG2NXFP47DITI4TBT...|1619896396472|           0|             true|
|   3.0|Could’ve been per...|This product is a...|    []|B07ZPC9QD4| B07ZPC9QD4|AGGW4YDY5F5V3EZG3...|1609363472452|           0|             true|
|   5.0|Good price for th...|Amazon offered th...|    []|B079QHML21| B07GZFM1ZM|AE2WG4JSNRRV5DLUD...|1619457976091|           0|             true|
|   5.0|Lightweight easy ...|No problems with ...|    []|B07J4XWXTR| B07J4XWXTR|AET6XQ5CVN6E4N7VV...|1640895274341|   

##**Select Text + Rating**

In [9]:
df = df.select(
    col("text").alias("review_text"),
    col("rating").alias("stars")
)

df = df.withColumn("label", col("stars") - 1)

##**Split Train/Test**

In [10]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

##**NLP Pipeline**

In [11]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml import Pipeline

tokenizer = Tokenizer(inputCol="review_text", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered")
hashingTF = HashingTF(inputCol="filtered", outputCol="rawFeatures", numFeatures=20000)
idf = IDF(inputCol="rawFeatures", outputCol="features")

pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf])

model = pipeline.fit(train_df)

train_data = model.transform(train_df)
test_data = model.transform(test_df)

##**Persist Features**

In [12]:
train_data.persist()
test_data.persist()

train_data.count()

160270

**Save Feature Data**

In [13]:
train_data.write.mode("overwrite").parquet("data/train_features")
test_data.write.mode("overwrite").parquet("data/test_features")

#**3_model_training**

##**Load Feature Data**

In [14]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

train_data = spark.read.parquet("data/train_features")
test_data = spark.read.parquet("data/test_features")

##**Logistic Regression**

In [15]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)
lr_model = lr.fit(train_data)

lr_model.write().overwrite().save("models/lr_model")

##**Naive Bayes**

In [16]:
from pyspark.ml.classification import NaiveBayes

nb = NaiveBayes(featuresCol="features", labelCol="label")
nb_model = nb.fit(train_data)

nb_model.write().overwrite().save("models/nb_model")

##**Random Forest**

In [17]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50)
rf_model = rf.fit(train_data)

rf_model.write().overwrite().save("models/rf_model")

##**SVM (OneVsRest)**

In [18]:
from pyspark.ml.classification import LinearSVC, OneVsRest

lsvc = LinearSVC(featuresCol="features", labelCol="label")
ovr = OneVsRest(classifier=lsvc)

svm_model = ovr.fit(train_data)

svm_model.write().overwrite().save("models/svm_model")

##**Cross Validation Example (LR)**

In [19]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(metricName="accuracy")

paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5]) \
    .build()

crossval = CrossValidator(estimator=lr,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)

cv_model = crossval.fit(train_data)

#**4_evaluation**

##**Load Test Data + Models**

In [20]:
from pyspark.sql import SparkSession
from pyspark.ml.classification import LogisticRegressionModel

spark = SparkSession.builder.getOrCreate()

test_data = spark.read.parquet("data/test_features")
lr_model = LogisticRegressionModel.load("models/lr_model")

##**Predictions**

In [21]:
lr_predictions = lr_model.transform(test_data)

##**Accuracy**

In [22]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(metricName="accuracy")
accuracy = evaluator.evaluate(lr_predictions)

print("Logistic Regression Accuracy:", accuracy)

Logistic Regression Accuracy: 0.6574125346086082


##**Confusion Matrix**

In [23]:
lr_predictions.groupBy("label", "prediction").count().show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  2.0|       0.0|  424|
|  1.0|       1.0|  261|
|  3.0|       2.0|  526|
|  4.0|       2.0|  702|
|  0.0|       1.0|  384|
|  0.0|       4.0|  753|
|  1.0|       0.0|  523|
|  2.0|       2.0|  523|
|  3.0|       1.0|  212|
|  2.0|       3.0|  582|
|  1.0|       4.0|  502|
|  4.0|       4.0|22264|
|  2.0|       4.0| 1093|
|  3.0|       4.0| 3480|
|  2.0|       1.0|  230|
|  1.0|       2.0|  297|
|  0.0|       0.0| 1651|
|  1.0|       3.0|  171|
|  4.0|       3.0| 2126|
|  0.0|       2.0|  309|
+-----+----------+-----+
only showing top 20 rows


##**Scalability Timing**

In [24]:
import time

start = time.time()
_ = lr_model.transform(test_data).count()
end = time.time()

print("Prediction Time:", end - start)

Prediction Time: 0.23399782180786133


#**Export Data from Spark for Tableau**

##**Export Model Performance Metrics**

In [25]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd

models = {
    "Logistic Regression": lr_model.transform(test_data),
    "Naive Bayes": nb_model.transform(test_data),
    "Random Forest": rf_model.transform(test_data),
    "SVM": svm_model.transform(test_data)
}

results = []

evaluator_acc = MulticlassClassificationEvaluator(metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(metricName="f1")

for name, preds in models.items():
    results.append({
        "Model": name,
        "Accuracy": evaluator_acc.evaluate(preds),
        "F1 Score": evaluator_f1.evaluate(preds)
    })

results_df = pd.DataFrame(results)
results_df.to_csv("model_performance.csv", index=False)

###**Export Confusion Matrix Data**

In [26]:
conf_df = lr_predictions.groupBy("label", "prediction").count().toPandas()
conf_df.to_csv("confusion_matrix.csv", index=False)

##**Export Business Insights Data**

In [27]:
from pyspark.sql.functions import length

business_df = df.withColumn("review_length", length("review_text")) \
                .groupBy("stars") \
                .avg("review_length") \
                .toPandas()

business_df.to_csv("business_insights.csv", index=False)

##**Export Scalability Data**

In [28]:
import time

scalability_results = []

for partitions in [100, 200, 400]:
    spark.conf.set("spark.sql.shuffle.partitions", partitions)

    start = time.time()
    _ = lr_model.transform(test_data).count()
    end = time.time()

    scalability_results.append({
        "Partitions": partitions,
        "Execution_Time": end - start
    })

pd.DataFrame(scalability_results).to_csv("scalability_results.csv", index=False)